# Restricted-input method reference
Source: gps_analysis/residence_identification/03-residence_distance.ipynb
Requires independently obtained inputs and source directory layout. Not an end-to-end pipeline. Outputs can contain sensitive data; keep the working directory private. External absolute paths have been replaced by external_inputs/<basename>. See docs/REPRODUCIBILITY.md.


In [ ]:
import os
from pathlib import Path
private_root = Path(os.environ["UFUG_PRIVATE_ROOT"]).resolve()
os.chdir(private_root)


In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.ops import nearest_points

# plot the distance distribution graph
import matplotlib.pyplot as plt

import os
# Working directory is explicitly configured in the first cell.


In [ ]:
park_boundary = gpd.read_file("gis_files/parks_allthree_boundaries.geojson")

park_boundary

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
import numpy as np

# If SciPy is available, we'll use gaussian_kde; otherwise fall back to a simple
# moving-average smoother on the histogram density.
try:
    from scipy.stats import gaussian_kde
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False

year = "2022"
parks_list = ['jurong', 'amk', 'punggol']
colours = ['#a6d854', '#66c2a5', '#b3b3b3']

# Consistent bins and x-grid
bins = np.arange(0, 10001, 100)          # 0–25 km, 50 m bins
x_eval = np.linspace(0, 10000, 800)      # x-grid for smooth lines

plt.figure(figsize=(10, 6))

for park, colour in zip(parks_list, colours):
    parks = park_boundary[park_boundary['layer'] == park]
    residences_gdf = gpd.read_file(
        f"gps_analysis/activities_identification_example/calibrated_residence_{park}_{year}_updated.geojson"
    ).to_crs("EPSG:3414")
    parks = parks.to_crs("EPSG:3414")

    # Distance to park boundary
    d = residences_gdf.geometry.distance(parks.geometry.union_all())
    d = d[np.isfinite(d)]  # guard against NaNs 

    # Save back (optional)
    residences_gdf["distance_to_park"] = d
    residences_gdf.to_file(
        f"gps_analysis/activities_identification_example/calibrated_residence_{park}_{year}_with_distance.geojson",
        driver="GeoJSON"
    )

    residences_gdf = residences_gdf[residences_gdf['distance_to_park'] <= 10000] # keep only distances ≤ 10 km
    d = d[d <= 10000]   # keep only distances ≤ 10 km

    # Histogram as proportions (density=True) with shared bins
    plt.hist(
        d, bins=bins, density=True,
        color=colour, alpha=0.35, label=None
    )

    # Smoothed line (KDE if SciPy; else smoothed histogram)
    if HAS_SCIPY and len(d) > 1:
        kde = gaussian_kde(d)
        y_smooth = kde(x_eval)
        # Clip tiny negatives from numerical noise
        y_smooth = np.clip(y_smooth, 0, None)
        plt.plot(x_eval, y_smooth, color=colour, lw=2, label=park.title())
    else:
        # Fallback: smooth the histogram density with a moving average
        hist, edges = np.histogram(d, bins=bins, density=True)
        centres = 0.5 * (edges[:-1] + edges[1:])
        # Simple moving average (window=3 or 5 depending on data size)
        window = 5 if len(hist) >= 5 else max(3, len(hist) // 2 * 2 + 1)
        kernel = np.ones(window) / window
        smooth = np.convolve(hist, kernel, mode="same")
        plt.plot(centres, np.clip(smooth, 0, None), color=colour, lw=2, label=park.title())

# Shared formatting
plt.title('')
plt.xlabel('Distance (metres)')
plt.ylabel('Proportion')
plt.xlim(0, 10000)
plt.ylim(0, 0.00125)
plt.grid(axis='y', alpha=0.75)
plt.legend(title="Park", frameon=False)
plt.tight_layout()

plt.savefig("gps_analysis/activities_identification_example/distance_to_park_all_2022_kde.svg", dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
import numpy as np

# If SciPy is available, we'll use gaussian_kde; otherwise fall back to a simple
# moving-average smoother on the histogram density.
try:
    from scipy.stats import gaussian_kde
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False

year = "2022"
parks_list = ['jurong', 'amk', 'punggol']
colours = ['#a6d854', '#66c2a5', '#b3b3b3']

# Consistent bins and x-grid
bins = np.arange(0, 15001, 500)          # 0–25 km, 500 m bins
x_eval = np.linspace(0, 15000, 800)      # x-grid for smooth lines

plt.figure(figsize=(10, 6))

for park, colour in zip(parks_list, colours):
    parks = park_boundary[park_boundary['layer'] == park]
    residences_gdf = gpd.read_file(
        f"gps_analysis/activities_identification_example/calibrated_residence_{park}_{year}_updated.geojson"
    ).to_crs("EPSG:3414")
    parks = parks.to_crs("EPSG:3414")

    # Distance to park boundary
    d = residences_gdf.geometry.distance(parks.geometry.union_all())
    d = d[np.isfinite(d)]  # guard against NaNs

    # Save back (optional)
    residences_gdf["distance_to_park"] = d
    residences_gdf.to_file(
        f"gps_analysis/activities_identification_example/calibrated_residence_{park}_{year}_with_distance.geojson",
        driver="GeoJSON"
    )

    # Histogram as proportions (density=True) with shared bins
    plt.hist(
        d,
        bins=bins,
        density=True,
        histtype="step",      # <- outline only
        linewidth=1.3,
        color=colour,
        label=None            # we will label using the KDE line
    )

    if HAS_SCIPY and len(d) > 1:
        kde = gaussian_kde(d)
        y_smooth = kde(x_eval)
        y_smooth = np.clip(y_smooth, 0, None)
        plt.plot(
            x_eval,
            y_smooth,
            color=colour,
            lw=2,
            label=park.title()    # legend from the line
        )
    else:
        hist, edges = np.histogram(d, bins=bins, density=True)
        centres = 0.5 * (edges[:-1] + edges[1:])
        window = 5 if len(hist) >= 5 else max(3, len(hist) // 2 * 2 + 1)
        kernel = np.ones(window) / window
        smooth = np.convolve(hist, kernel, mode="same")
        plt.plot(
            centres,
            np.clip(smooth, 0, None),
            color=colour,
            lw=2,
            label=park.title()
        )

# Shared formatting
plt.title('')
plt.xlabel('Distance (metres)')
plt.ylabel('Proportion of residents')
plt.xlim(0, 15000)
plt.ylim(0, 0.00125)
plt.grid(axis='y', alpha=0.75)
plt.legend(title="Park", frameon=False)
plt.tight_layout()

#plt.savefig("gps_analysis/activities_identification_example/distance_to_park_all_2022_kde.svg", dpi=300)
plt.show()